Creating the webdriver

In [ ]:
import email_info as ei
import smtplib
from email.message import EmailMessage
from email.utils import formataddr
import random
from json import JSONDecodeError

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
import hashlib
import json
from datetime import date, timedelta

In [ ]:


url="https://jobs.lever.co/waabi"

chrome_options = webdriver.ChromeOptions()
chrome_options.add_experimental_option("detach", True)

driver=webdriver.Chrome(options=chrome_options)

driver.get(url=url)
driver.maximize_window()



In [ ]:
with open("db.json","r") as f:
    try:
        db=json.load(f)
    except JSONDecodeError:
        db={}

# print(type(db))
# print(db.keys())

In [ ]:
def get_text_or_none(parent, by, value):
    elems = parent.find_elements(by, value)
    return elems[0].text.strip() if elems else "unknown"

def sha256_hex(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()[:10]

#db={}


In [ ]:
job_titles=driver.find_elements(By.CLASS_NAME,value="posting-title")
current_jobs=[]
new_job_alert=[]
old_job_filled_alert=[]

for index,job_title in enumerate(job_titles):
    url=job_title.get_attribute("href")
    job_id=sha256_hex(url)
    #print(f"{url}\t{job_id}")
    current_jobs.append(job_id)
    if job_id not in db.keys():
    # print(f"{url}\n{job_id}")
        db[job_id]=dict()
        # db[job_id]["index"]=index
        db[job_id]["job_name"]=get_text_or_none(job_title,By.CSS_SELECTOR,"h5")
        db[job_id]["work_policy"]=get_text_or_none(job_title,By.CSS_SELECTOR,".workplaceTypes")
        db[job_id]["work_policy"]=get_text_or_none(job_title,By.CSS_SELECTOR,".workplaceTypes")[:-2]
        db[job_id]["location"]=get_text_or_none(job_title,By.CSS_SELECTOR,".location")
        db[job_id]["commitment"]=get_text_or_none(job_title,By.CSS_SELECTOR,".commitment")
        db[job_id]["posted_date"]=date.today().strftime("%a %d-%b-%Y")
        db[job_id]["filled_date"]=""
        db[job_id]["url"]=url
        new_job_alert.append(job_id)

        # print(json.dumps(db[job_id], indent=4))


    # if index>3:
    #     break


In [ ]:
for old_job in db.keys():
    if old_job not in current_jobs:
        db[old_job]["filled_date"]=(date.today() - timedelta(days=1)).strftime("%a %d-%b-%Y")
        old_job_filled_alert.append(old_job)

In [ ]:
# print(json.dumps(db,indent=4))
print(f"{old_job_filled_alert= }\n{new_job_alert= }")

In [ ]:
if old_job_filled_alert:
    info=""
    for filled_job in old_job_filled_alert:
        info+=db[filled_job]["job_name"]
        info+="\n"


    msg = EmailMessage()
    #msg["From"] = ei.email
    msg["From"]= formataddr(("Ghulam Samdani",ei.email))
    msg["To"] = ei.receivers_email
    msg["Subject"] = f"Waabi Filled Positions"
    msg.set_content(info)

    with smtplib.SMTP(ei.host_address,ei.port_address) as connection:
        connection.starttls()
        connection.login(user=ei.email,password=ei.password)
        connection.set_debuglevel(1)
        connection.send_message(msg)

In [ ]:
for new_job in new_job_alert:
    #getting the jd
    driver.switch_to.new_window('tab')
    driver.get(url=url)

    info = ""
    descriptions = driver.find_elements(By.CLASS_NAME, value="section-wrapper")
    for desc in descriptions:
        info += desc.text


    msg = EmailMessage()
    #msg["From"] = ei.email
    msg["From"]= formataddr(("Ghulam Samdani",ei.email))
    msg["To"] = ei.receivers_email
    msg["Subject"] = f"Waabi {db[new_job]['job_name']}"
    msg.set_content(info)

    with smtplib.SMTP(ei.host_address,ei.port_address) as connection:
        connection.starttls()
        connection.login(user=ei.email,password=ei.password)
        connection.set_debuglevel(1)
        connection.send_message(msg)


In [ ]:
with open("db.json","w") as f:
    json.dump(db,f,indent=4)
driver.quit()


In [ ]:
url='https://efds.fa.em5.oraclecloud.com/hcmUI/CandidateExperience/en/sites/CX_1/jobs'
url=('https://efds.fa.em5.oraclecloud.com/hcmRestApi/resources/latest/recruitingCEJobRequisitions?onlyData=true&expand=requisitionList.workLocation,requisitionList.otherWorkLocations,requisitionList.secondaryLocations,flexFieldsFacet.values,requisitionList.requisitionFlexFields&'
     'finder=findReqs;'
     'siteNumber=CX_1,'
     'facetsList=LOCATIONS%3BWORK_LOCATIONS%3BWORKPLACE_TYPES%3BTITLES%3BCATEGORIES%3BORGANIZATIONS%3BPOSTING_DATES%3BFLEX_FIELDS,limit=25,lastSelectedFacet=LOCATIONS,selectedLocationsFacet=300000000425151,sortBy=POSTING_DATES_DESC')
params={
    "lastSelectedFacet" : "LOCATIONS",
    'mode': "location",
    "selectedLocationsFacet" :'300000000425151'
}

In [ ]:
import requests as rq
resp=rq.get(url=url)#,params=params)
print(resp)
print(resp.raise_for_status())



In [ ]:
import json
dat=resp.json()
print(json.dumps(dat,indent=4))

In [ ]:
print(type(dat['items'][0]['requisitionList']))

In [ ]:
s=dat['items'][0]['requisitionList']
print(type(s))
print(s)

In [ ]:
for x in s:
    print(type(x))
    print(x)
    print("******************")

In [ ]:
url='https://efds.fa.em5.oraclecloud.com/hcmUI/CandidateExperience/en/sites/CX_1/jobs/preview/59474/?lastSelectedFacet=LOCATIONS&mode=location&selectedLocationsFacet=300000000425151'
url = "https://efds.fa.em5.oraclecloud.com/hcmRestApi/resources/latest/recruitingCEJobRequisitions"
params = {
    "onlyData": "true",
    "expand": "requisitionList.workLocation,"
              "requisitionList.otherWorkLocations,"
              "requisitionList.secondaryLocations,"
              "flexFieldsFacet.values,"
              "requisitionList.requisitionFlexFields",
    "finder": (
        "findReqs;"
        "siteNumber=CX_1,"
        "facetsList=LOCATIONS;WORK_LOCATIONS;WORKPLACE_TYPES;"
        "TITLES;CATEGORIES;ORGANIZATIONS;POSTING_DATES;FLEX_FIELDS,"
        "limit=100,"
        "lastSelectedFacet=LOCATIONS,"
        "selectedLocationsFacet=300000000425151,"
        "sortBy=POSTING_DATES_DESC"
    )
}


resp=rq.get(url,params=params)
print(resp)
print(resp.raise_for_status())

In [ ]:
dat1=resp.json()
print(json.dumps(dat1,indent=4))

In [ ]:
from utils import sha256_hex
current_jobs_id=[]
job_data = {}
job_list=dat['items'][0]['requisitionList']
for job in job_list:
    # print(json.dumps(job,indent=4))
    hash_id=sha256_hex(job["Id"]+job["Title"]+job["PostedDate"])
    # print(hash_id)
    current_jobs_id.append(hash_id)
    job_data[hash_id]= {
        "job_id" : job["Id"],
        "job_name": job["Title"],
        "source": "self.name",
        "work_policy": job["WorkplaceType"],
        "location": job['PrimaryLocation'],
        "posted_date":job['PostedDate']
        "filled_date": "",
        "url": "url"
    }
    print(job_data[hash_id])
    print("***********************")


In [ ]:
url='https://efds.fa.em5.oraclecloud.com/hcmUI/CandidateExperience/en/sites/CX_1/job/51078'
url='https://efds.fa.em5.oraclecloud.com/hcmUI/CandidateExperience/en/sites/CX_1/jobs/preview/59474/?lastSelectedFacet=LOCATIONS&mode=location&selectedLocationsFacet=300000000425151'
url="https://efds.fa.em5.oraclecloud.com/hcmRestApi/resources/latest/recruitingCEJobRequisitionDetails?expand=all&onlyData=true&finder=ById;Id=%2258750%22,siteNumber=CX_1"
resp=rq.get(url=url)
print(resp.raise_for_status())
dat=resp.json()
print(json.dumps(dat,indent=4))


In [ ]:
base_domain="https://efds.fa.em5.oraclecloud.com"
url = (base_domain +
       "/hcmRestApi/resources/latest/recruitingCEJobRequisitionDetails")
params={
        "onlyData": "true",
        "expand": 'all',
        "finder":(
            "ById;"
            "Id = 58750,"
            "siteNumber=CX_1,"
        )
}

In [ ]:
resp=rq.get(url=url,params=params)
# print(resp.raise_for_status())
dat=resp.json()
print(json.dumps(dat,indent=4))

In [ ]:
yo=dat['items'][0]
print(type(dat))
print(json.dumps(yo,indent=4))


In [ ]:
info=dat['items'][0]["ExternalQualificationsStr"]+"\n\n"+dat['items'][0]["InternalQualificationsStr"]+"\n\n"+dat['items'][0]["InternalResponsibilitiesStr"]+"\n\n"+dat['items'][0]["ExternalDescriptionStr"]
# print(info)

In [ ]:
import re
new_info=re.sub(r"<.*?>","",info)
print(new_info)

In [ ]:
url='https://jobs.lever.co/kepler'
url='https://jobs.lever.co/kepler?location=Toronto%2C%20Ontario'
url='https://api.lever.co/v0/postings/kepler?mode=json'
params={
    'location':'Toronto, Ontario'
}
resp=rq.get(url=url,params=params)
resp.raise_for_status()
yo=resp.json()
print(json.dumps(yo,indent=4))